In [1]:
import os
import numpy as np
from scipy import stats
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import math
import scipy

In [2]:

from factor_analyzer import FactorAnalyzer

def compute_cct(dataframe, n_factors = 3, return_ev = False):
    
    '''
    Function returns the cultural consensus theory answer for a given dataset [emotion tracking dataset]
    
    Parameters
    ----------
    dataframe: pandas dataframe where each column is an individual observers response
    n_factors: number of factors to fit to dataframe
    return_ev: when set to True, returns the eigenvalue ratio between the first and second factor
    
    Notes
    -----
    When the eigenvalue ratio is >= 3, we can assume that their is a correct answer [1]. This suggests
    that factor 1 explains 3 times the variance in the data than factor 2. However, we assume that
    there is a correct answer for any given video so we do not look into the eigenvalue ratio right now.
    
    We set number of factors as 3 as default. There should only be one correct answer but the answer
    could be bi-modal resulting in 2 factors. Thus, three factors (three correct answers) is a 
    conservative number.
    
    References
    ----------
    [1] https://journals.sagepub.com/doi/10.1177/1525822X07303502
    
    
    Returns
    -------
    default:returns the cct consensus
    if return_ev == True: returns the cct consensus and the eigenvalue ratio between factor 1 and factor 2
        
    '''
    
    # create factor analysis
    fa = FactorAnalyzer(rotation = None)
    fa.fit(dataframe, n_factors) # fit to data, with n number of factors
    
    # Check Eigenvalues
    ev, v = fa.get_eigenvalues() # get eigenvales (measure of explained variance by each factor)
    ev_ratio = ev[0]/ev[1] # eigenvalue ratio between factor 1 and 2, should be >= 3.
    
    # CCT mean
    cct_m = fa.transform(dataframe)[:,0] # get weighted average of the dataset using factor 1
    
    # check if returning eigenvalue ratio
    if return_ev:
        return cct_m, ev_ratio 
    
    else:
        return cct_m 
    
    
def compute_ewe(dataframe):

    w = []
    s = dataframe.columns

    # get correlation of each subject to average
    for s in dataframe.columns:

        r1 = dataframe.drop(columns = [s]).mean(axis = 1)
        r,p = stats.pearsonr(r1,dataframe[s])

        w.append(r)

    ws = dataframe*np.array(w)
    ws = ws.mean(axis = 1)*1/sum(np.array(w))
    
    return(ws)
    
def normalize_corr(signal, a = -1, b = 1):
    
    A = signal.min().min()
    B = signal.max().max()
    C = (a-b)/(A-B)
    k = (C*A - a)/C
    return (signal-k)*C

def converging_model_fit_v2(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[(n+j)*2 for n in range(n)] for j in range(m)}))
    
    df = pd.DataFrame(scipy.signal.correlate2d(x,c))
    df = normalize_corr(df)
    
    nr, nc = df.shape # number of rows and columns
    cr = nr // 2
    cc = nc // 2

    cv = df.iloc[cr, cc] # center value   
    
    return(cv)

def diverging_model_fit_v2(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[-(n+j)*2 for n in range(n)] for j in range(m)}))
    
    df = pd.DataFrame(scipy.signal.correlate2d(x,c))
    df = normalize_corr(df)
    
    nr, nc = df.shape # number of rows and columns
    cr = nr // 2
    cc = nc // 2

    cv = df.iloc[cr, cc] # center value   
    
    return(cv)

def neighbor_model_fit_v2(x):
    n = len(x)
    m = len(x.columns)
    
    c = normalize_corr(pd.DataFrame.from_dict({j:[-abs(j-n) for n in range(n)] for j in range(m)}))
    
    df = pd.DataFrame(scipy.signal.correlate2d(x,c))
    df = normalize_corr(df)
    
    nr, nc = df.shape # number of rows and columns
    cr = nr // 2
    cc = nc // 2

    cv = df.iloc[cr, cc] # center value   
    
    return(cv)

# code to remove blinks and artifacts from the data

def outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xz[xz > z] = np.nan # remove values above 3 z score
    xz[xz < -z] = np.nan # remove values below -3 z score
    xz = xz.fillna(method='ffill') # change nans to previous value
    xz = xz.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xz

def emp_outlier_remover(x, z = 4):
    
    xc = x.replace(9999,np.nan) # remove blinks
    xz = (xc - xc.mean())/xc.std() # z-score
    xc[xz > z] = np.nan # remove values above 3 z score
    xc[xz < -z] = np.nan # remove values below -3 z score
    xc = xc.fillna(method='ffill') # change nans to previous value
    xc = xc.fillna(method = 'backfill') # change any nans at the beginning 
    
    return xc

def corrmean(data):
    return np.tanh(np.nanmean(np.arctanh(data)))

def corrmedian(data):
    return np.tanh(np.nanmedian(np.arctanh(data)))

def bootstrap_mean_corr(data,lo_ci = 2.5, hi_ci = 97.5,reps = 5000):
    
    ''' Computes bootstrapped mean and 95% confidence intervals with fisher z trasformation for correlations'''
    n = len(data)
    data = np.array((data))
    
    boot_data = [data[np.random.choice(range(n), size=n, replace=True)] for x in range(reps)] # get n datasets
    means = [np.tanh(np.nanmean(np.arctanh(x))) for x in boot_data] # get mean of each dataset
    
    m = np.tanh(np.nanmean(np.arctanh(means))) # fisher z transform
    ci = np.nanpercentile(means, [lo_ci, hi_ci]) # confidence intervals
    
    return m, ci

def bootstrap_mean(data,lo_ci = 2.5, hi_ci = 97.5,reps = 5000):
    
    ''' Computes bootstrapped mean and 95% confidence intervals with fisher z trasformation for correlations'''
    n = len(data)
    data = np.array((data))
    
    boot_data = [data[np.random.choice(range(n), size=n, replace=True)] for x in range(reps)] # get n datasets
    means = [np.nanmean(x) for x in boot_data] # get mean of each dataset
    
    m = np.nanmean(means)  
    ci = np.nanpercentile(means, [lo_ci, hi_ci]) # confidence intervals
    
    return m, ci

In [9]:
import os
import cv2

# get video lengths
videoPath = '/home/Jeff/1.0 projects/intersubject gaze correlations/videos'
os.chdir(videoPath)

video_lengths = {}

# Define common video extensions
video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv'}

# Iterate through files in the directory
for filename in os.listdir(videoPath):
    if os.path.isfile(filename):
        _, ext = os.path.splitext(filename)
        if ext.lower() in video_extensions:
            try:
                cap = cv2.VideoCapture(filename)
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                video_lengths[filename] = frames
                cap.release()
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                video_lengths[filename] = None  # Or handle as needed

print(f"Extracted frame counts for {len(video_lengths)} video files.")

Extracted frame counts for 8 video files.


In [10]:
import re

# Rename keys in video_lengths to extract identification number (e.g., 'Exp3_005_context-only.mp4' -> '005', '032SoloC_contextOnly.mp4' -> '032')
video_lengths_renamed = {}
for filename, frames in video_lengths.items():
    try:
        # First, try to extract digits after the first underscore
        match = re.search(r'_(\d+)', filename)
        if match:
            id_part = match.group(1)
        else:
            # Fallback: extract leading digits if no underscore
            match = re.match(r'(\d+)', filename)
            if match:
                id_part = match.group(1)
            else:
                id_part = filename  # Final fallback
        video_lengths_renamed[id_part] = frames
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        video_lengths_renamed[filename] = frames  # Fallback to original key

video_lengths_renamed

{'032': 1500,
 '005': 1681,
 '023': 1500,
 '011': 2064,
 '031': 1500,
 '020': 1674,
 '040': 1500,
 '004': 2369}

In [11]:
# preprocessing 
dataDir = '/home/Jeff/1.0 projects/intersubject gaze correlations/data/experiment 2/'
os.chdir(dataDir)

gazeData = {}

# iterate through folders

for folder in os.listdir():
    
    os.chdir(dataDir + folder)

    # identify and store core files
    gazeFile = [x for x in os.listdir() if 'gaze' in x][0]
    ratingFile = folder + '.csv'

    ## gaze data
    df = pd.read_csv(gazeFile).dropna().replace(9999,np.nan).iloc[:,1:].dropna()

    if len(df) ==0:
        continue
    
    for video in df.columns:
        if video not in gazeData:
            gazeData[video] = pd.DataFrame()

        v = video.split('_')[0]
        r = scipy.signal.resample(df[video],video_lengths_renamed[v])
        r = df[video]
        gazeData[video] = pd.concat([gazeData[video],pd.Series(r).rename(folder)],axis = 1)

In [3]:
questionnaireDir = '/home/Jeff/1.0 projects/intersubject gaze correlations/data/exp2_questionnaire_preprocessed.csv'
surveyDF = pd.read_csv(questionnaireDir)
surveyDF

,PID,Gender,Age,Born US,Education,EQ,AQ
0,BF083,Female,18,Yes,"High school graduate, diploma or the equivalen...",121,16
1,BF083,Female,18,Yes,"High school graduate, diploma or the equivalen...",125,17
2,BF056,Male,18,Yes,"High school graduate, diploma or the equivalen...",147,18
3,BF102,Male,18,Yes,"High school graduate, diploma or the equivalen...",117,13
4,BF026,Male,18,Yes,"High school graduate, diploma or the equivalen...",125,11
...,...,...,...,...,...,...,...
100,BF016,Female,32,No,"Some college credit, no degree",98,32
101,BF070,Female,32,No,"Master's degree (for example: MA, MS, MEng, ME...",136,21
102,BF071,Male,37,Yes,"Some college credit, no degree",155,26
103,BF010,Non-binary/non-conforming,37,Yes,"Associate degree (for example: AA, AS)",143,10


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from scipy.fft import fft, ifft
from scipy.signal import butter, filtfilt

# --- 1. Missing Values ---
def handle_missing_values(df: pd.DataFrame, method: str = "mean") -> pd.DataFrame:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if method == "mean":
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    return df

# --- 2. Outliers (Z-Score) ---
def handle_outliers(df: pd.DataFrame, method: str = "zscore") -> pd.DataFrame:
    if method != "zscore":
        return df
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        col_mean = df[col].mean()
        col_std = df[col].std()
        if col_std > 0:
            z_scores = (df[col] - col_mean) / col_std
            df.loc[np.abs(z_scores) > 3, col] = col_mean
    return df

# --- 3. Fourier Noise Reduction ---
def apply_fourier_denoise(df: pd.DataFrame, on: bool = True) -> pd.DataFrame:
    if not on:
        return df
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        data = df[col].values
        n = len(data)
        fhat = fft(data)
        psd = fhat * np.conj(fhat) / n
        threshold = 0.1 * np.max(psd).real
        indices = psd > threshold
        fhat_clean = fhat * indices
        df[col] = ifft(fhat_clean).real
    return df

# --- 4. Smoothing (Moving Average) ---
def smooth_moving_average(df: pd.DataFrame, on: bool, window: int = 5) -> pd.DataFrame:
    if not on or window <= 1:
        return df
    kernel = np.ones(window, dtype=np.float32) / window
    out = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for c in numeric_cols:
        out[c] = np.convolve(df[c].values, kernel, mode="same")
    return out

# --- 5. Low Pass Filter (Butterworth) ---
def low_pass_filter(df: pd.DataFrame, on: bool, cutoff: float = 30, fs: float = 75.0) -> pd.DataFrame:
    if not on:
        return df
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    nyq = 0.5 * fs
    normal_cutoff = max(min(cutoff / nyq, 0.99), 1e-6)
    b, a = butter(1, normal_cutoff, btype="low", analog=False)
    
    # We apply specifically to numeric columns to avoid issues with strings/dates
    arr = filtfilt(b, a, df[numeric_cols].values, axis=0)
    filtered_df = pd.DataFrame(arr, columns=numeric_cols, index=df.index, dtype=np.float32)
    
    # Update only the numeric columns in the original dataframe
    df[numeric_cols] = filtered_df
    return df

# --- 6. Normalization ---
def normalize(df: pd.DataFrame, method: str = "robust") -> pd.DataFrame:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if method == "robust":
        scaler = RobustScaler()
    elif method == "standard":
        scaler = StandardScaler()
    elif method == "minmax":
        scaler = MinMaxScaler()
    else:
        return df
        
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    return df

In [5]:
def gaze_preprocess(dataframe):
    d = dataframe.copy()
    d = handle_missing_values(d, "mean")
    d = handle_outliers(d, "zscore")
    d = normalize(d, "robust") 
    d = smooth_moving_average(d, on=True, window=int(75/10))
    d = low_pass_filter(d, on=True, cutoff=30, fs=25)
    return d

In [8]:
def get_gaze_on_video(gaze_h, screen_w=1280, screen_h=1024, scale=0.9):
    # 1. Video dimensions
    vid_w = screen_w * scale
    vid_h = screen_h * scale
    
    # 2. Top-left of the video in screen pixel coordinates
    vid_offset_x = (screen_w - vid_w) / 2
    vid_offset_y = (screen_h - vid_h) / 2
    
    # 3. Gaze in screen pixels (centered)
    gaze_x_px = gaze_h[0] * screen_h
    gaze_y_px = gaze_h[1] * screen_h
    
    # 4. Gaze relative to Video Top-Left (0,0)
    # Note: PsychoPy Y is positive UP, Video Y is positive DOWN
    video_x = gaze_x_px + (screen_w / 2) - vid_offset_x
    video_y = (screen_h / 2) - gaze_y_px - vid_offset_y
    
    return int(video_x), int(video_y)

# Example usage with a gaze point
current_gaze = (0.1, 0.1) # Height units
print(f"Pixel on video: {get_gaze_on_video(current_gaze)}")

{'011_gazeX':        BF087_2   BF083_2   BF090_1   BF007_1   BF089_1   BF008_1   BF011_1  \
 0    -0.034178 -0.108327 -0.008782 -0.043616  0.019189  0.009957 -1.175266   
 1    -0.053503 -0.148378 -0.018885 -0.064609  0.021588  0.011377 -1.565443   
 2    -0.071907 -0.191244 -0.026681 -0.085807  0.028328  0.012205 -1.958192   
 3    -0.092636 -0.231528 -0.037958 -0.107003  0.030585  0.014088 -2.348175   
 4    -0.132200 -0.288963 -0.060578 -0.146545  0.032936  0.000853 -2.740181   
 ...        ...       ...       ...       ...       ...       ...       ...   
 1688  0.789623  1.045900  0.548462  0.427235  1.995552  0.971773 -1.000763   
 1689  0.787070  1.009129  0.534464  0.420307  1.963650  1.068225 -1.002420   
 1690  0.784101  0.954356  0.520490  0.410866  1.936013  1.162423 -1.016016   
 1691  0.780846  0.811666  0.510687  0.402738  1.902003  1.259044 -0.991936   
 1692  0.776198  0.643091  0.498051  0.394438  1.873641  1.353199 -0.995867   
 
        BF012_1   BF086_1   BF092_1  

In [ ]:
import cv2
from collections import defaultdict

def plot_gaze_on_video(video_path, gaze_data_dict, output_dir=None, sample_every=5, video_id=None):
    """
    Overlay gaze trajectories from multiple participants on video frames.
    
    Parameters:
    -----------
    video_path : str
        Path to the video file.
    gaze_data_dict : dict
        Dictionary where keys are gaze column names (e.g., '001_gazeX', '001_gazeY') 
        and values are gaze coordinate arrays.
    output_dir : str, optional
        Directory to save annotated frames.
    sample_every : int
        Sample every N frames to avoid processing entire video.
    video_id : str, optional
        Identifier for the video (used in output filenames).
    """
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"Video: {video_path}")
    print(f"Frames: {total_frames}, FPS: {fps}, Resolution: {frame_width}x{frame_height}")
    
    # Extract unique participants from gaze data column names
    participants = defaultdict(dict)
    for col_name, gaze_values in gaze_data_dict.items():
        if '_gaze' in col_name:
            parts = col_name.split('_')
            participant_id = parts[0]
            coord_type = '_'.join(parts[1:])  # e.g., 'gazeX', 'gazeY'
            participants[participant_id][coord_type] = gaze_values
    
    print(f"Found {len(participants)} participants")
    
    # Color palette for different participants
    colors = [
        (255, 0, 0),    # Red
        (0, 255, 0),    # Green
        (0, 0, 255),    # Blue
        (255, 255, 0),  # Cyan
        (255, 0, 255),  # Magenta
        (0, 255, 255),  # Yellow
    ]
    
    frame_count = 0
    sampled_frames = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % sample_every == 0:
            # Create annotated frame
            annotated_frame = frame.copy()
            
            # Get frame index for gaze data lookup
            gaze_idx = frame_count
            
            # Plot gaze for each participant
            for participant_idx, (participant_id, gaze_coords) in enumerate(participants.items()):
                if 'gazeX' in gaze_coords and 'gazeY' in gaze_coords:
                    gaze_x_arr = gaze_coords['gazeX']
                    gaze_y_arr = gaze_coords['gazeY']
                    
                    # Check if we have data for this frame
                    if gaze_idx < len(gaze_x_arr) and gaze_idx < len(gaze_y_arr):
                        gaze_x = gaze_x_arr[gaze_idx]
                        gaze_y = gaze_y_arr[gaze_idx]
                        
                        # Convert to pixel coordinates (assuming gaze is in normalized coords)
                        # If gaze is already in pixels, skip this conversion
                        if not np.isnan(gaze_x) and not np.isnan(gaze_y):
                            # Assume gaze is normalized (-1 to 1 or 0 to 1)
                            # Convert to pixel coordinates
                            px_x = int((gaze_x + 1) / 2 * frame_width)  # Adjust based on your coordinate system
                            px_y = int((gaze_y + 1) / 2 * frame_height)
                            
                            # Clamp to frame bounds
                            px_x = max(0, min(px_x, frame_width - 1))
                            px_y = max(0, min(px_y, frame_height - 1))
                            
                            # Draw gaze point
                            color = colors[participant_idx % len(colors)]
                            cv2.circle(annotated_frame, (px_x, px_y), radius=8, color=color, thickness=2)
                            cv2.putText(annotated_frame, participant_id, (px_x + 10, px_y),
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            
            sampled_frames.append((frame_count, annotated_frame))
        
        frame_count += 1
    
    cap.release()
    
    # Display or save frames
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        for idx, (frame_num, frame) in enumerate(sampled_frames):
            output_path = os.path.join(output_dir, f"frame_{frame_num:06d}.jpg")
            cv2.imwrite(output_path, frame)
            print(f"Saved: {output_path}")
    else:
        # Display frames
        for idx, (frame_num, frame) in enumerate(sampled_frames[:5]):  # Show first 5 sampled frames
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f"Frame {frame_num} - Gaze Overlay")
            plt.axis('off')
            plt.tight_layout()
            plt.show()

# Example: Plot gaze for a specific video column
video_col = '001_gazeX'  # Example gaze column
video_id = video_col.split('_')[0]

# Find corresponding video
video_files = [f for f in os.listdir('/home/Jeff/1.0 projects/intersubject gaze correlations/videos') 
               if video_id in f or f.endswith('.mp4')]

if video_files:
    video_path = os.path.join('/home/Jeff/1.0 projects/intersubject gaze correlations/videos', video_files[0])
    
    # Extract gazeX and gazeY for all participants
    gaze_subset = {col: gazeData[col] for col in gazeData.columns if 'gaze' in col}
    
    # Plot gaze on video
    plot_gaze_on_video(video_path, gaze_subset, sample_every=10)
else:
    print(f"No video found for ID: {video_id}")